# 🧬 DiffBiophys Showcase: Differentiable Structure Refinement

Welcome to the **DiffBiophys** showcase! This notebook demonstrates how to use differentiable biophysics kernels in JAX to optimize protein structures directly against experimental (or synthetic) data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/diff-biophys/blob/main/examples/interactive_tutorials/diff_biophys_showcase.ipynb)

### What we'll do:
1. **Generate** a synthetic protein structure and its corresponding SAXS profile.
2. **Distort** the structure to create a starting \"guess\".
3. **Refine** the structure using Gradient Descent, visually tracking the loss and structural recovery.

---

In [ ]:
# Setup: Install DiffBiophys and visualization libraries
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q diff-biophys py3Dmol biotite matplotlib jax jaxlib
else:
    import os
    sys.path.insert(0, os.path.abspath("../"))

## 1. Initializing the Target State and Distorted State

First, we define a small 10-atom protein backbone structure (Target), calculate its SAXS profile, and then heavily distort its dihedral angles to create a poor starting state.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from diff_biophys.geometry import chain_nerf
from diff_biophys.saxs import debye_saxs

# Set up the system
n_atoms = 10
init_coords = jnp.array([[0.0, 0.0, 0.0], [1.5, 0.0, 0.0], [2.0, 1.4, 0.0]])
true_lengths = jnp.full(n_atoms - 3, 1.5)
true_angles = jnp.full(n_atoms - 3, jnp.radians(110.0))
true_dihedrals = jnp.array([1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0])

# 1. Create Target State
true_coords = chain_nerf(init_coords, true_lengths, true_angles, true_dihedrals)
q_values = jnp.linspace(0.01, 0.5, 30)
form_factors = jnp.ones((n_atoms, 30))
target_saxs = debye_saxs(true_coords, q_values, form_factors)

# 2. Distortion (add a constant offset to all dihedrals)
distorted_dihedrals = true_dihedrals + 0.5
initial_coords = chain_nerf(init_coords, true_lengths, true_angles, distorted_dihedrals)
initial_saxs = debye_saxs(initial_coords, q_values, form_factors)

print("Target and Distorted States Initialized.")

## 2. Gradient Descent Optimization

We define a simple Mean Squared Error (MSE) loss between the calculated SAXS profile of the current dihedral angles and the target SAXS profile. Because `diff-biophys` is built on JAX, we can simply `jax.grad` this function to get exact analytic gradients and use gradient descent!

In [ ]:
def loss_fn(dihedrals):
    coords = chain_nerf(init_coords, true_lengths, true_angles, dihedrals)
    s_profile = debye_saxs(coords, q_values, form_factors)
    s_loss = jnp.mean((s_profile - target_saxs)**2)
    return s_loss

grad_fn = jax.jit(jax.value_and_grad(loss_fn))

params = distorted_dihedrals
lr = 0.05
loss_history = []

for i in range(150):
    loss, grads = grad_fn(params)
    params -= lr * grads
    loss_history.append(float(loss))

print(f"Final Loss: {loss_history[-1]:.6f}")

# Calculate final optimized structure and its SAXS profile
final_coords = chain_nerf(init_coords, true_lengths, true_angles, params)
final_saxs = debye_saxs(final_coords, q_values, form_factors)

## 3. Visualize the Refinement Process

Let's look at the loss trajectory and compare the SAXS profiles. Notice how the final SAXS profile perfectly overlaps with the target!

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss Trajectory
ax1.plot(loss_history, 'k-', lw=2)
ax1.set_title("Optimization Loss Trajectory")
ax1.set_xlabel("Gradient Descent Step")
ax1.set_ylabel("MSE Loss")
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Plot 2: SAXS Profiles
ax2.plot(q_values, target_saxs, 'k-', lw=3, label="Target")
ax2.plot(q_values, initial_saxs, 'r--', lw=2, label="Initial (Distorted)")
ax2.plot(q_values, final_saxs, 'g--', lw=2, label="Final (Optimized)")
ax2.set_title("SAXS Scattering Profiles")
ax2.set_xlabel("q (Å⁻¹)")
ax2.set_ylabel("I(q)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 3D Structure Comparison

Finally, let's export the coordinates and use `py3Dmol` to visually prove that the refined structure matches the target structure.

In [ ]:
import numpy as np
import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import py3Dmol

def array_to_pdb(coords_jnp, filename):
    coords = np.array(coords_jnp)
    L = coords.shape[0]
    array = struc.AtomArray(L)
    array.coord = coords
    array.atom_name = ["CA"] * L
    array.element = ["C"] * L
    array.res_id = np.arange(1, L + 1)
    array.res_name = ["GLY"] * L
    array.chain_id = ["A"] * L
    
    pdb_file = pdb.PDBFile()
    pdb.set_structure(pdb_file, array)
    pdb_file.write(filename)
    with open(filename, "r") as f:
        return f.read()

target_str = array_to_pdb(true_coords, "target.pdb")
initial_str = array_to_pdb(initial_coords, "initial.pdb")
final_str = array_to_pdb(final_coords, "final.pdb")

view = py3Dmol.view(width=900, height=400, linked=False, viewergrid=(1,3))

view.addModel(target_str, "pdb", viewer=(0,0))
view.setStyle({'model': -1}, {'line': {'color': 'gray', 'linewidth': 5}}, viewer=(0,0))
view.addLabel("Target", {'position': {'x':0, 'y':0, 'z':0}, 'fontColor': 'black'}, viewer=(0,0))

view.addModel(initial_str, "pdb", viewer=(0,1))
view.setStyle({'model': -1}, {'line': {'color': 'red', 'linewidth': 5}}, viewer=(0,1))
view.addLabel("Initial", {'position': {'x':0, 'y':0, 'z':0}, 'fontColor': 'red'}, viewer=(0,1))

view.addModel(final_str, "pdb", viewer=(0,2))
view.setStyle({'model': -1}, {'line': {'color': 'green', 'linewidth': 5}}, viewer=(0,2))
view.addLabel("Final", {'position': {'x':0, 'y':0, 'z':0}, 'fontColor': 'green'}, viewer=(0,2))

view.zoomTo()
view.show()

## 5. Static 3D Plot (Fallback)

In environments where `py3Dmol` does not render, here is a static Matplotlib comparison of the structures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig = plt.figure(figsize=(15, 5))

c_target = np.array(true_coords)
c_init = np.array(initial_coords)
c_final = np.array(final_coords)

ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(c_target[:,0], c_target[:,1], c_target[:,2], marker='o', color='gray')
ax1.set_title('Target')

ax2 = fig.add_subplot(132, projection='3d')
ax2.plot(c_init[:,0], c_init[:,1], c_init[:,2], marker='o', color='red')
ax2.set_title('Initial (Distorted)')

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot(c_final[:,0], c_final[:,1], c_final[:,2], marker='o', color='green')
ax3.set_title('Final (Optimized)')

plt.tight_layout()
plt.show()
